In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Extract ZIP file
from pathlib import Path
import zipfile

# locate the ZIP file in google drive
zip_path = Path("/content/drive/MyDrive/MMA3001_Project/dataset/pork_rasher_v4.zip")

# Extract to Colab's local storage for faster processing
extract_path = Path("/content/pork_rasher_dataset")

if not zip_path.exists():
    print("Error: ZIP file not found at the specified path.")
    print("Please check if the file exists and if Google Drive is mounted correctly.")
else:
    extract_path.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_path)

    print("Dataset extracted successfully!")

    # Display the extracted folder structure
    print("\nExtracted files and folders:")
    for item in sorted(extract_path.iterdir()):
        print(item.name)

Dataset extracted successfully!

Extracted files and folders:
README.dataset.txt
README.roboflow.txt
data.yaml
test
train
valid


In [ ]:
# 2. read data.yaml, confirmation for 5 defect classes
import yaml

# locate the dataset configuration
dataset_path = Path("/content/pork_rasher_dataset")
yaml_path = dataset_path / "data.yaml"

# read the configuration
with open(yaml_path, "r") as file:
  config = yaml.safe_load(file)

# Display the configuration
print("Dataset configuration:\n")
for key, value in config.items():
  print(f"{key}: {value}")

Dataset configuration:

train: ../train/images
val: ../valid/images
test: ../test/images
nc: 5
names: ['loose-meat', 'packaging-error', 'twisted-meat', 'unsealed', 'wrinkle']
roboflow: {'workspace': 'hello-2aqe0', 'project': 'pork-rasher-error-packaging', 'version': 4, 'license': 'CC BY 4.0', 'url': 'https://universe.roboflow.com/hello-2aqe0/pork-rasher-error-packaging/dataset/4'}


In [ ]:
# 3. count images and annotations, so we know if the data split is reasonable
from collections import Counter

dataset_path = Path("/content/pork_rasher_dataset")
class_names = config["names"]

# check each dataset split
for split in ["train", "valid", "test"]:
  image_dir = dataset_path / split / "images"
  label_dir = dataset_path / split / "labels"

  # count images
  images = [
      p for p in image_dir.iterdir()
      if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
  ]

  # count bounding box annotations
  class_counts = Counter()
  for label_path in label_dir.iterdir():
    with open(label_path, "r") as file:
      for line in file:
        if line.strip():
          class_id = int(line.split()[0])
          class_counts[class_id] += 1

  # display results
  print(f"\n======{split.upper()} SET======")
  print("Total images:", len(images))

  for class_id, class_name in enumerate(class_names):
    print(f"{class_name}: {class_counts[class_id]}")


======TRAIN SET======
Total images: 3349
loose-meat: 66
packaging-error: 1712
twisted-meat: 72
unsealed: 1055
wrinkle: 18

======VALID SET======
Total images: 120
loose-meat: 3
packaging-error: 34
twisted-meat: 3
unsealed: 81
wrinkle: 0

======TEST SET======
Total images: 80
loose-meat: 3
packaging-error: 23
twisted-meat: 4
unsealed: 49
wrinkle: 0


In [ ]:
# Since the dataset is highly imbalanced, we need to address this issue
# 4. Create cleaned dataset to avoid source-image leakage
import shutil

original = dataset_path
clean = Path("/content/pork_rasher_dataset_clean")

# Copy original dataset into a fresh folder
shutil.rmtree(clean, ignore_errors=True)
shutil.copytree(original, clean)

def source_ids(split):
    return {
        p.name.split(".rf.")[0]
        for p in (clean / split / "images").glob("*.jpg")
    }

# Remove overlapping images and their labels
for split, blocked in [
    ("train", source_ids("valid") | source_ids("test")),
    ("valid", source_ids("test"))
]:
    for image in (clean / split / "images").glob("*.jpg"):
        if image.name.split(".rf.")[0] in blocked:
            image.unlink()
            (clean / split / "labels" / f"{image.stem}.txt").unlink()

# Update YOLO dataset configuration
config.update({
    "path": str(clean),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images"
})

(clean / "data.yaml").write_text(yaml.safe_dump(config))

# Confirm no shared source identifiers
assert not (
    source_ids("train") & source_ids("valid")
    or source_ids("train") & source_ids("test")
    or source_ids("valid") & source_ids("test")
)

print("No overlapping source identifiers between dataset splits.")

No overlapping source identifiers between dataset splits.


In [ ]:
# 5. Imbalance data split, investigate the rare wrinkle class

wrinkle_id = str(class_names.index("wrinkle"))
wrinkle_sources = Counter()
wrinkle_images = 0

for label in (clean / "train" / "labels").glob("*.txt"):

    # Count wrinkle annotations in each image
    count = sum(
        line.split()[0] == wrinkle_id
        for line in label.read_text().splitlines()
        if line.strip()
    )

    if count > 0:
        wrinkle_images += 1
        source = label.stem.split(".rf.")[0]
        wrinkle_sources[source] += count

print("Wrinkle annotations:", sum(wrinkle_sources.values()))
print("Images containing wrinkles:", wrinkle_images)
print("Unique source identifiers:", len(wrinkle_sources))
print("Annotations per source:", dict(wrinkle_sources))

Wrinkle annotations: 18
Images containing wrinkles: 18
Unique source identifiers: 6
Annotations per source: {'frame_with_red_8644_jpg': 3, 'frame_with_red_10412_jpg': 3, 'frame_with_red_10465_jpg': 3, 'frame_with_red_10498_jpg': 3, 'frame_with_red_10436_jpg': 3, 'frame_with_red_10552_jpg': 3}


In [ ]:
# Reserve one wrinkle source for exploratory validation

holdout_source = "frame_with_red_8644_jpg"

images = list(
    (clean / "train" / "images").glob(f"{holdout_source}.rf.*")
)

assert len(images) == 3

# Move images and their labels together
for image in images:
    label = clean / "train" / "labels" / f"{image.stem}.txt"

    shutil.move(image, clean / "valid" / "images" / image.name)
    shutil.move(label, clean / "valid" / "labels" / label.name)

print("Wrinkle source reserved for validation.")

Wrinkle source reserved for validation.


In [ ]:
# 6. Verify cleaned images and annotations

for split in ["train", "valid", "test"]:

    images = list((clean / split / "images").glob("*.jpg"))
    counts = Counter()

    for image in images:
        label = clean / split / "labels" / f"{image.stem}.txt"
        assert label.exists(), f"Missing label: {image.name}"

        for line in label.read_text().splitlines():
            if not line.strip():
                continue

            values = list(map(float, line.split()))
            assert len(values) == 5, f"Invalid label: {label.name}"

            class_id, x, y, w, h = values

            # Verify class ID and bounding-box coordinates
            assert class_id.is_integer() and 0 <= class_id < len(class_names)
            assert w > 0 and h > 0
            assert 0 <= x - w/2 <= x + w/2 <= 1
            assert 0 <= y - h/2 <= y + h/2 <= 1

            counts[int(class_id)] += 1

    # Display final results
    print(f"\n{split.upper()}: {len(images)} images")
    for i, name in enumerate(class_names):
        print(f"{name}: {counts[i]}")

print("\nAnnotation verification complete.")


TRAIN: 3310 images
loose-meat: 66
packaging-error: 1679
twisted-meat: 72
unsealed: 1052
wrinkle: 15

VALID: 122 images
loose-meat: 3
packaging-error: 34
twisted-meat: 3
unsealed: 80
wrinkle: 3

TEST: 80 images
loose-meat: 3
packaging-error: 23
twisted-meat: 4
unsealed: 49
wrinkle: 0

Annotation verification complete.


In [ ]:
# 7. Load the trained YOLOv8 baseline model
%pip install -q ultralytics

from ultralytics import YOLO

model_path = Path(
    "/content/drive/MyDrive/MMA3001_Project/model_runs/"
    "yolov8n_baseline/weights/best.pt"
)

# Load the saved model without retraining
model = YOLO(str(model_path))

print("Baseline model loaded successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 8.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Baseline model loaded successfully!


In [ ]:
# 8. Evaluate the trained baseline on unseen test images

test_results = model.val(
    data=str(clean / "data.yaml"),
    split="test",
    imgsz=640,
    batch=16,
    device=0,
    plots=True,
    project="/content/drive/MyDrive/MMA3001_Project/model_runs",
    name="yolov8n_baseline_test"
)

print("Baseline test evaluation complete!")

Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 3,006,623 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1254.7±622.6 MB/s, size: 65.2 KB)
val: Scanning /content/pork_rasher_dataset_clean/test/labels... 80 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 1.3Kit/s 0.1s
val: New cache created: /content/pork_rasher_dataset_clean/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s
                   all         80         79      0.565       0.49      0.533      0.266
            loose-meat          2          3          1      0.606      0.731      0.329
       packaging-error         16         23      0.498      0.518      0.497      0.124
          twisted-meat          4          4          0          0     0.0612     0.0247
              unsealed         49         49       0.7